# GPU Memory Benchmark for Process Reward Models

Benchmark GPU memory usage for the model weights of
process reward models (PRMs) across multiple candidates
and precision settings.

Each model is loaded with HF Transformers, measured for
GPU memory usage, and then unloaded before the next
model is tested. This keeps the comparison focused on
how model family and size affect GPU memory usage.

PRMs are loaded via `AutoModel` (not
`AutoModelForCausalLM`) because they expose a reward
head rather than a language-model head.

The benchmark explicitly loads models with
`dtype="float16"`. Without this setting,
`from_pretrained` may default to fp32, which can report
roughly twice the expected memory usage. Note: some PRM
checkpoints are published in bf16; V100 (sm_70) does not
support bf16, so fp16 is used throughout.

In [1]:
import os
os.environ["VLLM_CONFIGURE_LOGGING"] = "0"
import logging
logging.basicConfig(format='%(message)s', level=logging.FATAL+1)
logging.disable(logging.CRITICAL)

import warnings
warnings.filterwarnings("ignore")

import gc
import sys
sys.path.append("..")

import torch
from transformers import AutoModel, AutoTokenizer, AutoConfig

from unittests.notebook_utils import gpu_mem_used_gb

In [2]:
base_dir = '/groups/chichengz/tnn/datasets/'

model_names = [
    "Qwen2.5-Math-PRM-7B",
    "Llama3.1-8B-PRM-Deepseek-Data",
    # "Skywork-Reward-V2-Llama3.2-3B",
    # "Skywork-Reward-V2-Llama3.2-8B",
]

### Measure with HuggingFace Transformers

In [3]:
tf_results = []

for name in model_names:
    print(f"\n=== {name} ===")
    llm_dir = base_dir + name

    tokenizer = AutoTokenizer.from_pretrained(
        llm_dir, trust_remote_code=True,
    )
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id
        tokenizer.pad_token = tokenizer.eos_token

    config = AutoConfig.from_pretrained(
        llm_dir, trust_remote_code=True,
    )
    config.pad_token_id = tokenizer.pad_token_id

    llm_tf = AutoModel.from_pretrained(
        llm_dir,
        config=config,
        dtype="float16",
        device_map="cuda:0",
        trust_remote_code=True,
    )
    llm_tf.eval()

    mem_gb = gpu_mem_used_gb()
    print(f'  memory: {mem_gb:.2f} GB')
    tf_results.append((name, mem_gb))

    del llm_tf, tokenizer, config
    gc.collect()
    torch.cuda.empty_cache()

print("\n=== Summary (transformers) ===")
print(f"{'model':<35} {'memory (GB)':>12}")
print("-" * 48)
for name, mem in tf_results:
    print(f"{name:<35} {mem:>12.2f}")


=== Qwen2.5-Math-PRM-7B ===


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

  memory: 13.87 GB

=== Llama3.1-8B-PRM-Deepseek-Data ===


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

  memory: 14.56 GB

=== Summary (transformers) ===
model                                memory (GB)
------------------------------------------------
Qwen2.5-Math-PRM-7B                        13.87
Llama3.1-8B-PRM-Deepseek-Data              14.56
